# Digital Twin Demo — Counterfactual Simulation

Demonstrates the digital twin capability: generating patient-specific trajectories
and simulating counterfactual scenarios ("what if").

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.2)

## 1. Build the Hybrid Digital Twin Model

In [ ]:
from src.models.hybrid import HybridDigitalTwin

static = pd.read_parquet('../outputs/cohort/static_features.parquet')
ts = pd.read_parquet('../outputs/cohort/timeseries_processed.parquet')

# Use reduced epochs for demo (increase for production)
pipeline = HybridDigitalTwin(dgan_epochs=50, dgan_hidden_dim=64, dgan_noise_dim=32)
pipeline.fit(static.head(500), ts)
print('Model fitted.')

## 2. Generate a Synthetic Patient

In [ ]:
result = pipeline.generate(n_patients=1, seed=42)
patient_profile = result['static'].iloc[0]
trajectory = result['timeseries'][0]  # shape: (seq_len, n_features)

print('Patient Profile:')
for col in patient_profile.index:
    print(f'  {col}: {patient_profile[col]}')

In [ ]:
# Plot the patient's trajectory
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
hours = np.arange(trajectory.shape[0])

for ax, feat_idx, title in zip(axes.flat, [0, 1, 4, 5], 
                                 ['Feature 0', 'Feature 1', 'Feature 4', 'Feature 5']):
    if feat_idx < trajectory.shape[1]:
        ax.plot(hours, trajectory[:, feat_idx], 'b-', alpha=0.8)
        ax.set_xlabel('Hour')
        ax.set_title(title)
        ax.grid(True, alpha=0.3)

plt.suptitle('Synthetic Patient ICU Trajectory', fontsize=14)
plt.tight_layout()
plt.savefig('../outputs/figures/digital_twin_trajectory.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Counterfactual Simulation

What happens if we modify patient characteristics and regenerate trajectories?

In [ ]:
from src.simulation.counterfactual import CounterfactualSimulator

sim = CounterfactualSimulator(pipeline)
patient_dict = patient_profile.to_dict()

scenarios = {
    'Baseline': {},
    'Add AFib': {'has_afib': 1},
    'Younger (45y)': {'anchor_age': 45},
    'Add CKD + Diabetes': {'has_ckd': 1, 'has_diabetes': 1},
}

results = sim.compare_scenarios(patient_dict, scenarios, n_samples=20)
print(f'Generated {len(scenarios)} scenarios, each with 20 trajectory samples.')

In [ ]:
# Plot counterfactual comparison
fig, axes = plt.subplots(1, len(scenarios), figsize=(5*len(scenarios), 5))
colors = sns.color_palette('Set2', len(scenarios))

for ax, (name, color) in zip(axes, zip(scenarios.keys(), colors)):
    trajs = results[name]['trajectories']
    mean_traj = results[name]['mean_trajectory']
    std_traj = results[name]['std_trajectory']
    
    # Plot feature 0 (first vital sign)
    feat = 0
    if mean_traj.shape[1] > feat:
        hours = np.arange(mean_traj.shape[0])
        ax.plot(hours, mean_traj[:, feat], color=color, linewidth=2)
        ax.fill_between(hours, 
                        mean_traj[:, feat] - std_traj[:, feat],
                        mean_traj[:, feat] + std_traj[:, feat],
                        alpha=0.3, color=color)
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('Hour')
    ax.set_ylabel('Feature 0')

plt.suptitle('Counterfactual Comparison (Feature 0, mean +/- SD)', fontsize=14)
plt.tight_layout()
plt.savefig('../outputs/figures/counterfactual_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Individual Treatment Effect

Estimate the effect of adding atrial fibrillation to the patient profile.

In [ ]:
ite_result = sim.treatment_effect(
    patient_dict, 
    treatment={'has_afib': 1},
    n_samples=30
)

print('Individual Treatment Effect (AFib):')
print(f"  Factual outcome mean: {ite_result['y0_mean']:.4f} +/- {ite_result['y0_std']:.4f}")
print(f"  Counterfactual mean:  {ite_result['y1_mean']:.4f} +/- {ite_result['y1_std']:.4f}")
print(f"  ITE: {ite_result['ite']:.4f}")